In [1]:
import pyspark                                                                                                   
from pyspark.sql import SparkSession
spark = SparkSession.builder\
    .master("local[*]")\
    .appName('test')\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 22:56:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 22:56:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/09 22:56:17 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 22:56:43--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.165.71.48, 18.165.71.27, 18.165.71.56, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.165.71.48|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  2.28MB/s    in 23s     

2026-03-09 22:57:06 (2.89 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [4]:
df = spark.read\
    .option("header","true")\
    .parquet('yellow_tripdata_2025-11.parquet')

In [12]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [17]:
df.head(5)

[Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 2, 8, 11, 8), tpep_dropoff_datetime=datetime.datetime(2025, 11, 2, 8, 15, 21), passenger_count=1, trip_distance=1.24, RatecodeID=1, store_and_fwd_flag='N', PULocationID=186, DOLocationID=230, payment_type=1, fare_amount=7.9, extra=0.0, mta_tax=0.5, tip_amount=2.53, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=15.18, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 6, 14, 1, 48), tpep_dropoff_datetime=datetime.datetime(2025, 11, 6, 14, 25, 53), passenger_count=2, trip_distance=1.84, RatecodeID=1, store_and_fwd_flag='N', PULocationID=164, DOLocationID=237, payment_type=2, fare_amount=20.5, extra=0.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=25.25, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 

In [14]:
df \
    .repartition(4) \
    .write.parquet('data/pq/yellow_tripdata_2025-11/')

In [39]:
df = spark.read.parquet('data/pq/yellow_tripdata_2025-11/')

In [15]:
#Question 2. What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

> ls -lh data/pq/yellow_tripdata_2025-11/
total 101M
-rw-r--r-- 1 vscode vscode 25M Mar  9 23:16 part-00000-41116c9e-6dce-4a2b-9656-9c2e8f080ec6-c000.snappy.parquet
-rw-r--r-- 1 vscode vscode 25M Mar  9 23:16 part-00001-41116c9e-6dce-4a2b-9656-9c2e8f080ec6-c000.snappy.parquet
-rw-r--r-- 1 vscode vscode 25M Mar  9 23:16 part-00002-41116c9e-6dce-4a2b-9656-9c2e8f080ec6-c000.snappy.parquet
-rw-r--r-- 1 vscode vscode 25M Mar  9 23:16 part-00003-41116c9e-6dce-4a2b-9656-9c2e8f080ec6-c000.snappy.parquet

In [ ]:
#Question 3. How many taxi trips were there on the 15th of November? Answer: 162604

In [40]:
from pyspark.sql import functions as F
df.filter(F.col('tpep_pickup_datetime') >= '2025-11-15') \
    .filter(F.col('tpep_pickup_datetime') < '2025-11-16') \
  .count()

162604

In [ ]:
#Question 4. What is the length of the longest trip in the dataset in hours? Answer: 90.6

In [41]:
from pyspark.sql import functions as F

df.withColumn(
    'trip_hours',
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
) \
.agg(F.max('trip_hours').alias('longest_trip_hours')) \
.show()

+------------------+
|longest_trip_hours|
+------------------+
| 90.64666666666666|
+------------------+



In [27]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 23:41:28--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.244.84.132, 18.244.84.43, 18.244.84.178, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.244.84.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.007s  

2026-03-09 23:41:28 (1.80 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [29]:
df_zones = spark.read\
    .option("header","true")\
    .csv('taxi_zone_lookup.csv')

In [32]:
df_zones \
    .write.parquet('taxi_zone_lookup.parquet')

In [36]:
from pyspark.sql import types as T
df_zones = spark.read\
    .option("header", "true") \
    .parquet('taxi_zone_lookup.parquet')
df_zones = df_zones \
    .withColumn('LocationID', df_zones.LocationID.cast(T.IntegerType()))

In [37]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [ ]:
#Question 6. What is the name of the LEAST frequent pickup location Zone? Answer: Arden Heights

In [45]:
df \
    .groupBy('PULocationID') \
    .agg(F.count('*').alias('pickup_count')) \
    .join(df_zones, df.PULocationID == df_zones.LocationID) \
    .orderBy('pickup_count', ascending=True) \
    .select('Zone', 'pickup_count') \
    .show(1)

+-------------+------------+
|         Zone|pickup_count|
+-------------+------------+
|Arden Heights|           1|
+-------------+------------+
only showing top 1 row
